In [1]:
import pandas as pd
import glob
import os
import numpy as np

path = r"C:\Users\rebec\Documents\BigData\Final Project\Data"
all_files = glob.glob(os.path.join(path, "*.csv"))

df_list = [pd.read_csv(file) for file in all_files]
df = pd.concat(df_list, ignore_index=True)
display(df.head())

,STATION,LATITUDE,LONGITUDE,ELEVATION,NAME,month,day,hour,ANN-PRCP-NORMAL,meas_flag_ANN-PRCP-NORMAL,...,comp_flag_DJF-PRCP-AVGNDS-GE200HI,years_DJF-PRCP-AVGNDS-GE200HI,DJF-PRCP-AVGNDS-GE400HI,meas_flag_DJF-PRCP-AVGNDS-GE400HI,comp_flag_DJF-PRCP-AVGNDS-GE400HI,years_DJF-PRCP-AVGNDS-GE400HI,DJF-PRCP-AVGNDS-GE600HI,meas_flag_DJF-PRCP-AVGNDS-GE600HI,comp_flag_DJF-PRCP-AVGNDS-GE600HI,years_DJF-PRCP-AVGNDS-GE600HI
0,US1NCAG0001,36.4590,-81.1525,946.1,"SPARTA 3.5 SSW, NC US",99,99,99,56.25,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,US1NCCB0008,35.3647,-80.6550,193.9,"CONCORD 4.5 SW, NC US",99,99,99,46.99,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,US1NCCD0014,36.0546,-81.4843,477.9,"LENOIR 10.6 NNE, NC US",99,99,99,52.56,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,US1NCCH0004,35.6486,-79.3361,173.1,"GOLDSTON 3.8 N, NC US",99,99,99,48.94,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,US1NCCH0016,35.7721,-78.9981,89.6,"APEX 9.2 WNW, NC US",99,99,99,48.30,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df_filtered = df.iloc[:, np.r_[0:5, 8]]

display(df_filtered)

,STATION,LATITUDE,LONGITUDE,ELEVATION,NAME,ANN-PRCP-NORMAL
0,US1NCAG0001,36.4590,-81.1525,946.1,"SPARTA 3.5 SSW, NC US",56.25
1,US1NCCB0008,35.3647,-80.6550,193.9,"CONCORD 4.5 SW, NC US",46.99
2,US1NCCD0014,36.0546,-81.4843,477.9,"LENOIR 10.6 NNE, NC US",52.56
3,US1NCCH0004,35.6486,-79.3361,173.1,"GOLDSTON 3.8 N, NC US",48.94
4,US1NCCH0016,35.7721,-78.9981,89.6,"APEX 9.2 WNW, NC US",48.30
...,...,...,...,...,...,...
138,US1NCWY0002,35.4704,-78.0892,35.1,"PIKEVILLE 6.2 WSW, NC US",51.22
139,US1NCYD0003,36.0532,-80.6028,260.6,"YADKINVILLE 6.3 SSE, NC US",47.30
140,US1NCYD0004,36.1309,-80.6555,280.4,"YADKINVILLE 0.2 E, NC US",48.89
141,US1NCYN0004,35.8297,-82.3285,947.9,"BURNSVILLE 6.5 SSW, NC US",53.96


In [5]:
df_filtered = df_filtered.rename(columns={
    'STATION': 'station',
    'NAME': 'name',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'ELEVATION': 'elevation',
    'ANN-PRCP-NORMAL': 'annual_precip'})

nc_data = df_filtered

display(nc_data.head())

,station,latitude,longitude,elevation,name,annual_precip
0,US1NCAG0001,36.4590,-81.1525,946.1,"SPARTA 3.5 SSW, NC US",56.25
1,US1NCCB0008,35.3647,-80.6550,193.9,"CONCORD 4.5 SW, NC US",46.99
2,US1NCCD0014,36.0546,-81.4843,477.9,"LENOIR 10.6 NNE, NC US",52.56
3,US1NCCH0004,35.6486,-79.3361,173.1,"GOLDSTON 3.8 N, NC US",48.94
4,US1NCCH0016,35.7721,-78.9981,89.6,"APEX 9.2 WNW, NC US",48.30


In [8]:
def normalize(column):
    return (column - column.min()) / (column.max() - column.min())

nc_data['precip_norm'] = normalize(nc_data['annual_precip'])
nc_data['elev_norm'] = normalize(nc_data['elevation'])
nc_data['lat_norm'] = normalize(nc_data['latitude'])
nc_data['long_norm'] = normalize(nc_data['longitude'])

nc_data['coastal_penalty'] = 1 - nc_data['long_norm']

nc_data['survivability'] = (
    0.25 * nc_data['precip_norm'] +
    0.5 * nc_data['elev_norm'] +
    0.15 * nc_data['lat_norm'] +
    0.2 * nc_data['coastal_penalty']
)

def classify_survivability(score):
    if score >= 0.7:
        return 'High'
    elif score >= 0.4:
        return 'Medium'
    else:
        return 'Low'
    
nc_data['survivability_category'] = nc_data['survivability'].apply(classify_survivability)

In [9]:
nc_data.head()

,station,latitude,longitude,elevation,name,annual_precip,precip_norm,elev_norm,lat_norm,long_norm,coastal_penalty,survivability,survivability_category
0,US1NCAG0001,36.4590,-81.1525,946.1,"SPARTA 3.5 SSW, NC US",56.25,0.466767,0.809968,0.972794,0.355953,0.644047,0.796404,High
1,US1NCCB0008,35.3647,-80.6550,193.9,"CONCORD 4.5 SW, NC US",46.99,0.119039,0.165796,0.463011,0.412286,0.587714,0.299652,Low
2,US1NCCD0014,36.0546,-81.4843,477.9,"LENOIR 10.6 NNE, NC US",52.56,0.328201,0.409009,0.784403,0.318383,0.681617,0.540539,Medium
3,US1NCCH0004,35.6486,-79.3361,173.1,"GOLDSTON 3.8 N, NC US",48.94,0.192264,0.147983,0.595267,0.561626,0.438374,0.299023,Low
4,US1NCCH0016,35.7721,-78.9981,89.6,"APEX 9.2 WNW, NC US",48.30,0.168231,0.076475,0.652800,0.599898,0.400102,0.258236,Low


In [13]:
nc_data.to_csv(r'C:\Users\rebec\Documents\BigData\Final Project\Data\nc_survivability.csv', index=False)